In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("train.csv")

# Display the first five rows
df.head()

# Check dataset information
df.info()

# Create a data quality report
quality_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Data Types": df.dtypes,
    "Unique Values": df.nunique()
})

# Display duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

quality_report

# Check numeric values and their ranges
df.describe()

# Check unusual values
print("Negative Ages:", (df["Age"] < 0).sum())
print("Negative Fares:", (df["Fare"] < 0).sum())
print("Invalid Passenger Classes:", (~df["Pclass"].isin([1, 2, 3])).sum())
print("Invalid Survival Values:", (~df["Survived"].isin([0, 1])).sum())

# Save original dataset statistics for comparison
before_rows = len(df)
before_nulls = df.isnull().sum().sum()
before_duplicates = df.duplicated().sum()
before_dtypes = df.dtypes.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
Duplicate Rows: 0
Negative Ages: 0
Negative Fares: 0
Invalid Passenger Classes: 0
Invalid Survival Values: 0


#Missing Data Handling
I will use different strategies depending on the type of data. Age and Fare will be filled with their median values because they may contain extreme values. Embarked will be filled with its mode because it is categorical. Cabin will be removed because most of its values are missing.

In [ ]:
# Fill missing numerical values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Fill missing categorical values
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Remove Cabin because it has too many missing values
df.drop(columns=["Cabin"], inplace=True)

# Check remaining missing values
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
# Count duplicates before removing them
duplicates_before = df.duplicated().sum()

# Remove duplicate records
df.drop_duplicates(inplace=True)

# Check how many duplicates were removed
print("Duplicates Removed:", duplicates_before)
print("Remaining Duplicates:", df.duplicated().sum())

# Standardise gender values
df["Sex"] = df["Sex"].astype("string").str.strip().str.lower()

df["Sex"] = df["Sex"].replace({
    "male": "Male",
    "female": "Female",
    "m": "Male",
    "f": "Female"
})

# Standardise Embarked values
df["Embarked"] = df["Embarked"].astype("string").str.strip().str.upper()

# Remove unnecessary spaces from text columns
for col in ["Name", "Ticket"]:
    df[col] = df[col].astype("string").str.strip()

# Check standardised values
print(df["Sex"].unique())
print(df["Embarked"].unique())

# Identify outliers using the IQR method
numeric_columns = ["Age", "Fare"]

outlier_counts = {}

for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_limit) |
        (df[col] > upper_limit)
    ]

    outlier_counts[col] = len(outliers)

    print(f"\n{col} Outliers:", len(outliers))
    print("Lower Limit:", lower_limit)
    print("Upper Limit:", upper_limit)

# Display outlier counts
outlier_counts

# Convert passenger ID into string
df["PassengerId"] = df["PassengerId"].astype("string")

# Convert text columns into string
for col in ["Name", "Sex", "Ticket", "Embarked"]:
    df[col] = df[col].astype("string")

# Convert numerical columns into appropriate types
df["Survived"] = df["Survived"].astype("int64")
df["Pclass"] = df["Pclass"].astype("int64")
df["Age"] = pd.to_numeric(df["Age"])
df["SibSp"] = df["SibSp"].astype("int64")
df["Parch"] = df["Parch"].astype("int64")
df["Fare"] = pd.to_numeric(df["Fare"])

# Check corrected data types
df.dtypes

# Check missing values
print("Missing Values:", df.isnull().sum().sum())

# Check duplicates
print("Duplicate Rows:", df.duplicated().sum())

# Check final row count
print("Total Rows:", len(df))

# Check invalid values
print("Negative Ages:", (df["Age"] < 0).sum())
print("Negative Fares:", (df["Fare"] < 0).sum())

# Define expected data types
expected_dtypes = {
    "PassengerId": "string",
    "Survived": "int",
    "Pclass": "int",
    "Name": "string",
    "Sex": "string",
    "Age": "float",
    "SibSp": "int",
    "Parch": "int",
    "Ticket": "string",
    "Fare": "float",
    "Embarked": "string"
}

# Check how many columns have the correct data type
def dtype_accuracy(data):

    correct = 0

    for col, expected in expected_dtypes.items():

        actual = data[col].dtype

        if expected == "string" and pd.api.types.is_string_dtype(actual):
            correct += 1

        elif expected == "int" and pd.api.types.is_integer_dtype(actual):
            correct += 1

        elif expected == "float" and pd.api.types.is_float_dtype(actual):
            correct += 1

    return round(correct / len(expected_dtypes) * 100, 2)


# Calculate original data type accuracy
before_accuracy = dtype_accuracy(
    pd.read_csv("train.csv").astype({
        "PassengerId": "string",
        "Name": "string",
        "Sex": "string",
        "Ticket": "string",
        "Embarked": "string"
    })
)

# Calculate cleaned data type accuracy
after_accuracy = dtype_accuracy(df)

# Create comparison table
comparison = pd.DataFrame({
    "Before Cleaning": [
        before_rows,
        before_nulls,
        before_duplicates,
        f"{before_accuracy}%"
    ],

    "After Cleaning": [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        f"{after_accuracy}%"
    ]

}, index=[
    "Total Rows",
    "Null Values",
    "Duplicate Rows",
    "Data Type Accuracy"
])

comparison

# Save the cleaned dataset as a new CSV file
df.to_csv("titanic_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")


# Load and verify the cleaned file
cleaned_df = pd.read_csv("titanic_cleaned.csv")

cleaned_df.head()

Duplicates Removed: 0
Remaining Duplicates: 0
<StringArray>
['Male', 'Female']
Length: 2, dtype: string
<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: string

Age Outliers: 66
Lower Limit: 2.5
Upper Limit: 54.5

Fare Outliers: 116
Lower Limit: -26.724
Upper Limit: 65.6344
Missing Values: 0
Duplicate Rows: 0
Total Rows: 891
Negative Ages: 0
Negative Fares: 0
Cleaned dataset saved successfully!


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.0500,S
